# 03 — Model Evaluation

After training, this notebook runs the full evaluation suite:
- Confusion matrix
- ROC and Precision-Recall curves
- Error analysis (misclassified examples)
- Combined pipeline (ELA + CNN) score comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from PIL import Image

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.classifier import build_model
from src.ela import ELAAnalyzer
from src.utils import load_config

config = load_config('../config.yaml')
cfg = config['classifier']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

MODEL_PATH = '../models/classifier.pth'
TEST_DIR   = '../data/test'

if not Path(MODEL_PATH).exists():
    print(f'⚠  Model not found at {MODEL_PATH}. Run train_classifier.py first.')

## 3.1 — Load model and run inference

In [ ]:
transform = transforms.Compose([
    transforms.Resize((cfg['image_size'], cfg['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(TEST_DIR, transform=transform)
loader  = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
class_names = [k for k, _ in sorted(dataset.class_to_idx.items(), key=lambda x: x[1])]
print(f'Test set: {len(dataset)} images | classes: {dataset.class_to_idx}')

ckpt = torch.load(MODEL_PATH, map_location=device)
backbone = ckpt.get('backbone', cfg.get('backbone', 'resnet50'))
model = build_model(backbone=backbone, num_classes=2, pretrained=False).to(device)
model.load_state_dict(ckpt.get('model_state_dict', ckpt))
model.eval()
print(f'Loaded: {backbone}, trained for {ckpt.get("epoch", "?")} epochs, saved AUC={ckpt.get("val_auc", "?")}')

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in loader:
        logits = model(images.to(device))
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend((probs > 0.5).astype(int))
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)
print('Inference complete.')

## 3.2 — Classification report

In [ ]:
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
print(f'ROC-AUC  : {roc_auc_score(all_labels, all_probs):.4f}')
print(f'Avg Prec : {average_precision_score(all_labels, all_probs):.4f}')

## 3.3 — Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.5)
ax.set_ylabel('True', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 — ROC and PR curves

In [ ]:
auc = roc_auc_score(all_labels, all_probs)
ap  = average_precision_score(all_labels, all_probs)
fpr, tpr, _ = roc_curve(all_labels, all_probs)
prec, rec, _ = precision_recall_curve(all_labels, all_probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC = {auc:.3f}')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, lw=2, color='darkorange', label=f'AP = {ap:.3f}')
axes[1].fill_between(rec, prec, alpha=0.1, color='darkorange')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 — Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(all_probs[all_labels == 0], bins=40, alpha=0.6, color='#2ecc71', label='Real',   density=True)
ax.hist(all_probs[all_labels == 1], bins=40, alpha=0.6, color='#e74c3c', label='Forged', density=True)
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax.set_xlabel('Predicted Forgery Probability')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Class')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

overlap = np.mean((all_probs > 0.3) & (all_probs < 0.7))
print(f'Ambiguous predictions (0.3–0.7): {overlap*100:.1f}%  ← lower is better')

## 3.6 — Error analysis: misclassified examples

In [ ]:
image_paths = [s[0] for s in dataset.samples]

errors = [(image_paths[i], all_labels[i], all_preds[i], all_probs[i])
          for i in range(len(all_labels)) if all_labels[i] != all_preds[i]]

print(f'Misclassified: {len(errors)} / {len(all_labels)}  ({len(errors)/len(all_labels)*100:.1f}%)')

if errors:
    show = errors[:8]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for ax, (path, true, pred, prob) in zip(axes, show):
        img = Image.open(path).resize((224, 224))
        ax.imshow(np.array(img))
        true_name = class_names[true]
        pred_name = class_names[pred]
        ax.set_title(f'True: {true_name}\nPred: {pred_name} ({prob:.2f})',
                     color='red', fontsize=9)
        ax.axis('off')
    for ax in axes[len(show):]:
        ax.axis('off')
    plt.suptitle('Misclassified Samples', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3.7 — Combined pipeline: ELA + CNN

In [ ]:
ela = ELAAnalyzer(config['ela'])
ela_scores = [ela.get_forgery_score(p) for p in image_paths]

ela_w = config['inference']['ela_weight']
cls_w = config['inference']['classifier_weight']
combined_probs = ela_w * np.array(ela_scores) + cls_w * all_probs
combined_preds = (combined_probs > 0.5).astype(int)

cnn_auc      = roc_auc_score(all_labels, all_probs)
ela_auc      = roc_auc_score(all_labels, ela_scores)
combined_auc = roc_auc_score(all_labels, combined_probs)

print(f'CNN only  AUC: {cnn_auc:.4f}')
print(f'ELA only  AUC: {ela_auc:.4f}')
print(f'Combined  AUC: {combined_auc:.4f}  ← ensemble')

methods = ['ELA only', 'CNN only', 'Combined']
aucs    = [ela_auc, cnn_auc, combined_auc]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, aucs, color=['#3498db', '#e67e22', '#2ecc71'],
              edgecolor='black', width=0.4)
for bar, v in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.4f}',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('ROC-AUC')
ax.set_title('AUC: ELA vs CNN vs Combined Pipeline')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()